# 第一层：用户写的train_ultralytics.py

> 神经网络的训练分为4步：
  - 1：前向传播；
  - 2：计算误差；
  - 3：反向传播；
  - 4：参数更新。
> 理解神经网络的训练过程可以从这四步出发。

> 在`train_ultralytics.py`里，这四步被ultralytics封装成了`model.train()`方法，接下来分析一下这个方法。

In [ ]:
def train(
        self,
        trainer=None,
        **kwargs: Any,
    ):
        """使用指定的数据集和训练配置训练模型。

        该方法支持通过一系列可自定义的设置进行模型训练。它支持使用自定义训练器或默认训练方式进行训练。
        该方法会处理诸如从检查点恢复训练、与 Ultralytics HUB 集成，以及在训练后更新模型和配置等场景。

        使用 Ultralytics HUB 时，如果会话中已有加载的模型，该方法会优先使用 HUB 训练参数，并在提供了本地参数时发出警告。
        它会检查 pip 更新，并合并默认配置、方法特定默认值以及用户提供的参数来配置训练过程。

        参数：
            trainer (BaseTrainer, 可选): 用于模型训练的自定义训练器实例。如果为 None，则使用默认训练器。
            **kwargs (Any): 用于训练配置的任意关键字参数。常见选项包括：
                - data (str): 数据集配置文件路径。
                - epochs (int): 训练轮数。
                - batch (int): 训练批大小。
                - imgsz (int): 输入图像尺寸。
                - device (str): 运行训练的设备（例如 'cuda'、'cpu'）。
                - workers (int): 用于数据加载的工作线程数。
                - optimizer (str): 用于训练的优化器。
                - lr0 (float): 初始学习率。
                - patience (int): 在没有观察到明显改进时等待的轮数，用于提前停止训练。
                - augmentations (list[Callable]): 训练期间要应用的增强函数列表。

        返回：
            (ultralytics.utils.metrics.DetMetrics | None): 如果可用且训练成功，则返回训练指标；
                否则返回 None。具体指标类型取决于任务。

        示例：
            >>> model = YOLO("yolo26n.pt")
            >>> results = model.train(data="coco8.yaml", epochs=3)
        """
        self._check_is_pytorch_model()
        if hasattr(self.session, "model") and self.session.model.id:  # Ultralytics HUB session with loaded model
            if any(kwargs):
                LOGGER.warning("using HUB training arguments, ignoring local training arguments.")
            kwargs = self.session.train_args  # overwrite kwargs

        checks.check_pip_update_available()

        overrides = YAML.load(checks.check_yaml(kwargs["cfg"])) if kwargs.get("cfg") else self.overrides
        custom = {
            # NOTE: handle the case when 'cfg' includes 'data'.
            "data": (overrides.get("data") if kwargs.get("cfg") else None)
            or DEFAULT_CFG_DICT["data"]
            or TASK2DATA[self.task],
            "model": self.overrides["model"],
            "task": self.task,
        }  # method defaults
        args = {**overrides, **custom, **kwargs, "mode": "train", "session": self.session}  # prioritizes rightmost args
        if args.get("resume"):
            if args["resume"] is True:  # resume=True (boolean) uses current model as checkpoint
                if self.ckpt and self.ckpt.get("epoch", -1) >= 0 and self.ckpt.get("optimizer") is not None:
                    args["resume"] = self.ckpt_path
                else:
                    LOGGER.warning(
                        f"model '{self.ckpt_path}' is not a resumable training checkpoint "
                        f"(missing epoch/optimizer state). Use 'resume' only to continue incomplete training. "
                        f"Starting new training instead."
                    )
                    args["resume"] = False

        self.trainer = (trainer or self._smart_load("trainer"))(overrides=args, _callbacks=self.callbacks)
        if not args.get("resume") and self.ckpt:
            # Reuse the already-loaded checkpoint model to avoid re-resolving remote weight sources during trainer setup.
            self.trainer.model = self.trainer.get_model(weights=self.model, cfg=self.model.yaml)
            self.model = self.trainer.model

        self.trainer.train()
        # Update model and cfg after training
        if RANK in {-1, 0}:
            ckpt = self.trainer.best if self.trainer.best.exists() else self.trainer.last
            self.model, self.ckpt = load_checkpoint(ckpt)
            self.overrides = self._reset_ckpt_args(self.model.args)
            self.metrics = getattr(self.trainer.validator, "metrics", None)  # TODO: no metrics returned by DDP
        return self.metrics

> 上面的代码是`ultralytics/engine/model.py`第715行处的定义，我将英文注释翻译成了中文。我们来看一下入参：

- **其中第一个参数**是Python 实例方法的隐式参数，调用时不用写，会自己调用这个方法的实例。
  - 在`train_ultralytics.py`中，模型实例的初始化位于第140行和第190行在两次`model.train()`前面。语法为`model = YOLO(xxx)`

- **第二个参数**是`trainer`（训练器），它是对数据加载器、优化器、学习率调度器、混合精度、日志打印、验证评估、早停、保存权重等一大堆流程的封装。方法里默认采用`_smart_load（）`方法，自动寻找对应的训练器。
  - 训练器这个类在ultralytics 里采用两级继承设计：
    - **父类** `BaseTrainer`，管所有任务通用的逻辑。
    - **子类**：以目标检测任务为例，类型为 `DetectionTrainer`，只补齐目标检测任务特有的逻辑。

- **第三个参数**是`kwargs`(我的理解是argmuents key word)，他是一个字典（键值对的容器），键（超参数名称）是字符串，值（超参数的值）的类型是任意的。
  - 在`model.train()`方法里，`kwargs`会与`custom`，`overrides`这两个超参数字典合成一份最终版本的超参数字典`args`。（具体逻辑见下方代码）
  - 在后续训练器运行时，会把`default.yaml`（defalut.yaml是ultralytics里所有的模型和任务共用的一个超参数表）的所有键全部展开，校验传`args`的键名是否存在于`default.yaml`中：
    - 若不存在，则会抛出 `SyntaxError`；
    - 若存在但不是本次任务所需要的超参数，会静默处理，这一般不会引起错误；
    - 若存在且是本次任务所需要的超参数，超参数会优先采用你传入的值，否则采用默认值；


> 在train_ultralytics.py中一共有两次训练。第一次训练只让`Head`和`Neck`的权重变化，第二次训练让全网络权重变化，要理解原因，我们先简单了解一下YOLO模型：
- YOLO模型可以大致分为3层：
    - `Backbone`：它学到的是边缘、纹理、形状这类通用特征，与要识别的物体关系不大
    - `Neck`：负责把不同尺度的特征合在一起
    - `Head`：把合并的特征与要检测的N类物体的概率联系起来

<br>

> 对于第一次训练：
- 由于我们是在已经训练好的`yolo11s.pt`（COCO 80类）的基础上迁移训练识别9类食材，因此`Head`层是一定要修改的，而新修改的`Head`层的权重是完全随机的，一开始，随机检测头给出的梯度又大又乱。如果这时全网用**较大学习率**，这些梯度会一路传回`Backbone`，把COCO上已经学好的特征冲掉。等`Head`自己稳住时，前面的特征已经坏了。如果为了保护`Backbone`，从第一轮就用**很小的学习率**训练全网，`Head`又学得太慢。
- 因此第一次训练只更新`Head`和`Neck`，用较大的学习率，让新的`Head`尽快学会特征对应哪一种食材、框在哪里。`Neck`也更新是为了提供足够的自由度，让训练能够顺利的进行下去，只靠`Head`的那点自由度，很难把学到的特征对应到9类食材上。

<br>

> 对于第二次训练：
- 由于冰箱中食材的边缘，纹理等特征与COCO学到的特征有一定区别，因此第二次训练加载第一次训练时的best.pt，用较小的学习率对`Backbone`进行微调以适应YOLO模型的工作场景。`Head`和`Neck`也要一起微调，这涉及到了深度学习里的一个机理：**上下游层之间的表征耦合**（`Co-adaptation`）
    - 通俗的来说，`Co-adaptation`是指：神经网络里上游怎么“表达”，和下游怎么“理解”，是训练时互相迁就着一起长出来的，谁都离不开谁。
- 如果阶段二解冻 Backbone，却把 Neck 和 Head 锁死，会同时出现两个问题：
    - 特征分布漂移，上下游失配
    - 锁死下游，会反过来卡死 Backbone 的优化方向

train_ultralytics.py中调用的两次`model.train()`方法的入参都是第三个参数`** kwargs`，从`data=...到exist_ok=True`这19个关键字参数，参数名都不是 `trainer`，所以Python会把它们全部打包成一个字典传给`kwargs`。接下来我们看一下两次训练的入参都有什么以及入参的区别：

In [ ]:
"""
第一次训练：
"""
model = YOLO(pretrained_weights)

model.train(
    data=str(dataset_yaml),
    epochs=epochs,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=NUM_WORKERS,
    seed=SEED,
    amp=USE_AMP,
    optimizer=OPTIMIZER,
    lr0=FREEZE_LR,
    lrf=0.01,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    cos_lr=True,
    freeze=FREEZE_LAYERS,  # 底层在 BaseTrainer._setup_train 中冻结 model.0~model.10
    patience=PATIENCE,
    project=str(output_dir),
    name="stage1_freeze",
    exist_ok=True,
)
"""
第二次训练：
"""
model = YOLO(str(stage1_ckpt))

model.train(
    data=str(dataset_yaml),
    epochs=epochs,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=NUM_WORKERS,
    seed=SEED,
    amp=USE_AMP,
    optimizer=OPTIMIZER,
    lr0=FINETUNE_LR,
    lrf=0.01,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    cos_lr=True,
    freeze=None,  # 全网解冻，0~23 层全部参与梯度更新（.dfl 常量投影层除外）
    patience=PATIENCE,
    project=str(output_dir),
    name="stage2_finetune",
    exist_ok=True,
)

| 参数 | 这次脚本里的值 | 作用 |
|---|---|---|
| `data` | `dataset.yaml` 的路径 | 训练集、验证集路径和类别定义 |
| `epochs` | 见下方差异 | 最多训练多少轮 |
| `imgsz` | `640` | 输入边长，训练前会缩放到 640 |
| `batch` | `8` | 每个 step 的图片数 |
| `device` | 有 CUDA 时为 `0`，否则 `"cpu"` | 用哪块设备 |
| `workers` | `0` | DataLoader 子进程数。Windows 调试设成 0，避免多进程问题 |
| `seed` | `42` | 随机种子，让两次运行可复现 |
| `amp` | 有 CUDA 时为 `True` | 混合精度。显存更省，速度通常更快 |
| `optimizer` | `"AdamW"` | 优化器 |
| `lr0` | 见下方差异 | 初始学习率 |
| `lrf` | `0.01` | 最终学习率 = `lr0 × lrf`，也就是降到初始值的 1% |
| `weight_decay` | `5e-4` | AdamW 的权重衰减 |
| `warmup_epochs` | `2.0` | 前 2 个 epoch 学习率从很小逐渐升到 `lr0` |
| `cos_lr` | `True` | warmup 之后按余弦曲线从 `lr0` 降到 `lr0 × lrf` |
| `freeze` | 见下方差异 | 冻结哪些层，被冻结的层不更新权重 |
| `patience` | `15` | 验证集指标连续 15 个 epoch 不提升就早停 |
| `project` | `runs/ultralytics` | 结果根目录 |
| `name` | 见下方差异 | 这次 run 的子目录名 |
| `exist_ok` | `True` | 同名目录已存在时直接往里写，不自动改名 |


**四个不同的入参：**

| 参数 | 阶段一 `stage1_freeze` | 阶段二 `stage2_finetune` |
|---|---|---|
| `epochs` | `10`（`FREEZE_EPOCHS`） | `40`（`FINETUNE_EPOCHS`） |
| `lr0` | `1e-3`（`FREEZE_LR`） | `1e-4`（`FINETUNE_LR`） |
| `freeze` | `11` | `None` |
| `name` | `"stage1_freeze"` | `"stage2_finetune"` |

> 在上面的这么多参数中，对以下这些参数进行深入讨论，讨论时会引入几个概念：

- 1、**退火**：在神经网络训练中，指的是随着训练推进逐步降低学习率的策略。这么做的好处是：
    - 在**训练初期**，学习步子大，不容易卡在糟糕的局部极小值或鞍点附近；
    - 在**训练后期**，学习步子小，参数能稳稳落进损失函数的谷底，不会因为步长太大在最低点附近来回震荡。
    - ultralytics默认只有线性退火和余弦退火策略（设当前轮次为 $t \in [0, T]$，总训练轮数为 $T = \text{epochs}$，初始学习率为 $\eta_0 = \text{lr0}$，最终学习率倍率为 $\text{lrf}$，则最终学习率 $\eta_{\min} = \eta_0 \cdot \text{lrf}$）：
      - **线性退火**（对应`cos_lr = False`）：每一轮学习率按恒定斜率从 $\eta_0$ 匀速降到 $\eta_{\min}$：
        $$\eta_t = \eta_0 \left[ \left(1 - \frac{t}{T}\right)(1 - \text{lrf}) + \text{lrf} \right] = \eta_0 - (\eta_0 - \eta_{\min})\frac{t}{T}$$
        <br>
      - **余弦退火**（对应`cos_lr = True`，`train_ultralytics.py` 使用的策略）：按半个余弦周期（$0 \to \pi$）从 $\eta_0$ 平滑降到 $\eta_{\min}$，呈现“前期下降慢、中期下降快、后期再放缓”的曲线，兼顾前期探索与后期精细收敛：
        $$\eta_t = \eta_0 \left[ \text{lrf} + \frac{1 - \text{lrf}}{2}\left(1 + \cos\left(\frac{\pi t}{T}\right)\right) \right] = \eta_{\min} + \frac{1}{2}(\eta_0 - \eta_{\min})\left(1 + \cos\left(\frac{\pi t}{T}\right)\right)$$
        <br>
        由上面公式可知，余弦项里的相位进度是 $\frac{\pi t}{T}$（其中 $T = \text{epochs}$），相当于把从 $\eta_0$ 降到 $\eta_{\min}$ 的半个余弦周期固定拉伸到全部 `epochs` 上。
          - `epochs` 越大，单轮推进的相位 $\frac{\pi}{T}$ 越小，学习率衰减得**越慢**（高学习率探索期和低学习率精调期都会拉长）；
          - `epochs` 越小，单轮推进的相位越大，学习率会**更快**跌向 $\eta_{\min}$；

<br>

- 2、**预热**：在神经网络训练中，网络的学习率一开始不是`lr0`，而是在`warmup_epochs`这几轮里从接近0的学习率升到`lr0`。这么做的原因是（目前成熟的神经网络训练库大部分采用线性的方式进行预热）：
    - 层的权重初值是随机分配的，大部分情况下，初值对应的梯度和优化器统计（与参数更新有关，**梯度**只能说明参数往那变，优化器会根据**统计**决定实际变多少）不可靠，在这种情况下使用大的学习率容易导致权重向错误的方向一去不返，后续训练再多轮都无法回到正确的权重分布。

<br>

- 3、**优化器**：在神经网络训练中，负责第4步（**参数更新**）的算法，决定每个参数具体往哪个方向改、改多少。
    - 反向传播算出的**梯度**只表示当前这一步损失函数的变化方向；优化器会结合**学习率**、当前梯度以及自身维护的**优化器统计**（历史梯度信息），算出每个参数最终的更新量。
    - ultralytics的优化器有很多，我只学了下面这几个：
        - **SGD（随机梯度下降）**：基础做法是直接按“当前梯度 × 学习率”更新；实际训练中通常会加**动量（momentum）**，把历史梯度平滑叠加，减少震荡、加快收敛。
            - 基础 SGD（$\theta$ 为网络中正在被训练的权重，$g_t$ 为第 $t$ 步梯度，$\eta$ 为学习率）：
              $$\theta_t = \theta_{t-1} - \eta \, g_t$$
            - 带动量的 SGD（$\mu$ 为动量系数，决定优化器记得多少步的梯度，ultralytics上默认为0.9，大约记住最近的10步；$v_t$ 为累积的历史梯度速度）：
              $$v_t = \mu \, v_{t-1} + g_t, \qquad \theta_t = \theta_{t-1} - \eta \, v_t$$
              <br>
        - **Adam / AdamW**：除了记录历史梯度的滑动平均（决定方向），还会记录历史梯度平方的滑动平均，按参数分别缩放步长——梯度剧烈波动的参数步子变小，梯度平稳的参数步子相对更大；**AdamW** 在此基础上将**权重衰减**（`weight_decay`）与梯度更新解耦，是目前迁移学习与微调中很常用的选择（train_ultralytics.py使用的就是 `AdamW`）。
            - 一阶矩 $m_t$（梯度滑动平均）与二阶矩 $v_t$（梯度平方滑动平均）。
              - 默认 $\beta_1=0.9$，意思是新的$m_t$里90%是旧方向，10%是新方向，记忆范围是最近10步梯度。
              - $\beta_2=0.999$，衡量着这个参数的梯度平方在训练历史里的平均大小，记忆范围是最近1000步梯度。
              $$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t, \qquad v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
            - 偏差修正（修正初值 $m_0=0,\ v_0=0$ 在训练前期偏小的问题）：
              $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
            - **Adam** 更新（$\epsilon$ 为防止除零的小常数，如 $10^{-8}$；若加权重衰减，会并入 $g_t$ 一起被 $\sqrt{\hat{v}_t}$ 缩放）：
              $$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$
            - **AdamW** 更新（$\lambda$ 为 `weight_decay`，直接按学习率衰减权重，与自适应梯度项解耦）：
              $$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_{t-1}$$

<br>

- 4、**早退（早停 / Early Stopping）**：在神经网络训练时，每轮结束后都会在验证集上评估模型效果；如果验证集指标长时间不再提升，继续训练既浪费算力，又容易让模型在训练集上**过拟合**（训练损失继续下降，但验证集表现停滞甚至变差）。早停策略用于自动监测这一现象并在合适时机提前终止训练。
    - **Ultralytics 中的判定机制**（对应 `EarlyStopping` 类）：
        - 每轮验证结束后计算适应度分数 `fitness`（在当前版本目标检测任务中，`Metric.fitness` 对应验证集的 `mAP@0.5:0.95`），同时维护历史最高分 `best_fitness` 和取得该最高分的轮次 `best_epoch`。
        - 若本轮 `fitness > best_fitness`（或训练初期 `best_fitness == 0`），就更新 `best_fitness` 与 `best_epoch = epoch`，并把当前权重保存为 `best.pt`。
        - 接着计算距上次刷新已经过去多少轮：
          $$\Delta = \text{epoch} - \text{best\_epoch}$$
          <br>
    - **`patience` 如何影响早停**：
        - `patience` 表示**允许验证集指标连续不刷新的最大容忍轮数**，每轮检查条件 $\Delta \ge \text{patience}$：
            - 一旦连续 `patience` 轮没有超过历史最高分，就立即触发早停终止训练（最终保留的最优权重仍是 `best_epoch` 对应的 `best.pt`）；
            - 只要在满 `patience` 轮前有任意一轮刷新了 `best_fitness`，`best_epoch` 就会更新，计数 $\Delta$ 归零重新开始计算。
        - **取值权衡**：
            - `patience` **太小**：容易因为验证集指标的正常抖动，或学习率还没退火到合适区间就过早停训，错过后续收敛（欠拟合）；
            - `patience` **太大**：模型已经收敛或开始过拟合后仍会空转很多轮，浪费训练时间；
            - `patience = 0`：在 Ultralytics 中表示**关闭早停**（源码中将容忍轮数设为 $\infty$），强制跑满全部 `epochs`。
        - **结合 `train_ultralytics.py` 来看**：脚本里设置了 `PATIENCE = 15`。阶段一 `epochs = 10 < 15`，因此不会触发早停，会完整跑满 10 轮；阶段二 `epochs = 40`，若在第 $k$ 轮得到最优 `best.pt` 后连续 15 轮都没有再超过它，就会在第 $k + 15$ 轮提前结束训练。

<br>

- 5、**正则化（Regularization）**：**正则化**就是刻意给模型施加一定的约束或扰动，逼它去学真正通用的特征；以避免**过拟合（Overfitting）**。
    - **常见正则化手段与对应超参数**：

      | 正则化手段 | 对应的正则化超参数 | 控制的是什么 |
      |---|---|---|
      | **权重衰减（L2 正则）** | `weight_decay` | 权重往 `0` 收缩的力度，数值越大限制越强 |
      | **Dropout** | `dropout`（丢弃率，如 `0.2`） | 训练时随机让多大比例的神经元临时失活 |
      | **数据增强** | `mosaic`、`mixup`、`hsv_h/s/v` 等 | 给训练图片做随机裁剪、拼接、变色的强度 |
      | **早停（Early Stopping）** | `patience` | 验证集连续多少轮没提升就提前停止，防止继续练过头 |

    - 说明：`weight_decay` 并不是 `AdamW` 专用的参数，`SGD`、`Adam`、`AdamW` 等常见优化器都支持它；只是 `AdamW` 将权重衰减与自适应梯度更新解耦，避免了二阶矩缩放导致衰减失真。
    <br>
    - **确定正则化超参数值的原则**：
        - **1）先拟合、后正则（以训练/验证曲线为依据）**：先用默认或较弱的正则化跑通基线，观察训练集与验证集差距：
            - 训练集和验证集指标都差 → **欠拟合**，应**减弱**正则化（如调小 `weight_decay`、降低数据增强强度、减小或关闭 `dropout`）；
            - 训练集很好但验证集停滞或变差 → **过拟合**，再逐步**增强**正则化。
        - **2）匹配数据规模与任务阶段**：数据量越小、模型容量相对越大，越容易过拟合，需要更明显的正则化保护；反之数据量极大时可以适当放松。迁移学习微调阶段通常保留适度正则，避免过强扰动破坏预训练特征。
        - **3）警惕多种正则化的叠加效应，每次只调单一维度**：`weight_decay`、强数据增强、Dropout、早停同时拉满很容易把模型压成欠拟合。例如现代检测网络大量使用 BatchNorm 与 Mosaic/MixUp 增强，本身已有较强正则化效果，因此检测主干通常 `dropout = 0`。
        - **4）按数量级粗调，并注意与学习率、优化器的联动**：像 `weight_decay` 通常按对数尺度试探（如 $10^{-4}$、$5\times 10^{-4}$、$10^{-3}$、$10^{-2}$），而不是微调小数位；且每步实际衰减量与 $\eta \lambda$ 相关，切换优化器（如 `SGD` ↔ `AdamW`）或大幅改动学习率/Batch Size 时，需要重新核对量级。
    <br>
    - **工程上流行的做法（主流训练库实践）**：
        - **1）以成熟训练库的默认配方（Recipe）为起点**：
            - 以ultralytics（YOLO 系列）为例，直接沿用 `default.yaml` 中经过大规模验证的默认值（如 `weight_decay=5e-4`、`mosaic=1.0`、`dropout=0.0`），并在底层自动执行按有效 Batch Size 缩放 `weight_decay`、以及在训练最后10轮（`close_mosaic=10` ）关闭 Mosaic 增强以回归真实图像分布精调。
        - **2）参数分组衰减已成为全行业共识**：PyTorch 生态的几乎所有主流库（Ultralytics、timm、Hugging Face、OpenMMLab）在构建优化器时，都会把参数分组——**只对卷积层/线性层的 `weight` 施加 `weight_decay`，而对 BatchNorm/LayerNorm 参数及所有 `bias` 将 `weight_decay` 设为 `0`**，避免破坏归一化层尺度和偏置拟合能力。
        - **3）按收益优先级调整，必要时用自动化搜索（HPO）**：在目标检测等视觉任务中，工程调参优先级通常是 **数据质量与数据增强 > 学习率调度与早停 > `weight_decay` > `dropout`**；若默认配方遇瓶颈，常借助 **Ultralytics `model.tune()`**（演化搜索）或 **Optuna / Ray Tune / W&B Sweeps** 在对数空间里做小预算自动搜参，并配合实验日志（TensorBoard / W&B）对比验证曲线定稿。

> 在讲完训练工作后，train_ultralytics.py中剩下的工作都是辅助训练完成端到端的工作，下面简单介绍一下：

<br>

- 1、**参数检查（启动前自检）**：对应 `find_images()`、`resolve_dataset_config()` 与 `validate_ready()` 三个函数（第 63~122 行），在训练开始前先确认配置文件与数据就绪，避免进入训练循环后才报错。
    - **解析并校验 `dataset.yaml`**（`resolve_dataset_config`）：检查文件是否存在；校验必需字段 `train`、`val` 是否填写；检查类别字段 `names` / `nc` 是否存在且数量一致；将 `path`（相对路径会相对于 `dataset.yaml` 所在目录解析）与 `train`/`val` 拼接成绝对目录路径。
    - **扫描图片文件**（`find_images`）：递归扫描 `train_dir` 和 `val_dir` 下所有常见格式图片（`.jpg`/`.jpeg`/`.png`/`.bmp`/`.webp`，不区分大小写）并排序。
    - **就绪断言与环境摘要**（`validate_ready`）：若训练集或验证集图片数为 0，直接抛出 `RuntimeError` 并提示具体目录；通过后打印 `DEVICE`、`USE_AMP`、类别数及训练/验证集图片数量，并返回 `(train_images, val_images)` 供后续步骤复用。

<br>

- 2、**测试集推理与导出（推理验货与可选部署导出）**：对应 `infer_and_export()` 函数（第 223~256 行），在阶段二训练结束后对最终权重做一次快速验货，并按需导出部署格式。
    - **单图推理验货**（`best_model.predict(...)`）：加载阶段二产出的最优权重 `final_ckpt`（优先 `best.pt`，否则回退 `last.pt`），实际抽取验证集第一张图 `val_images[0]` 进行推理（注：脚本里未单独划分测试集，是用验证集样本做快速抽检）。核心推理参数包括输入尺寸 `imgsz=640`、置信度过滤阈值 `conf=0.25`（`CONF_THRES`）、NMS 非极大值抑制阈值 `iou=0.45`（`IOU_THRES`），并开启 `save=True` 将画好检测框的结果保存到 `runs/ultralytics/infer_check`，同时打印检测到的目标框数量 `len(results[0].boxes)`。
    - **可选 ONNX 导出**（`best_model.export(...)`）：由顶部开关 `EXPORT_ONNX`（默认 `False`）控制；设为 `True` 时会调用 `best_model.export(format="onnx", imgsz=640, simplify=True)`，将权重导出为经过计算图简化的 `.onnx` 文件，便于后续脱离 PyTorch 做跨平台推理部署。

# 第二层：ultralytics写的model.train()方法

In [ ]:
"""
这段代码来自train()方法内部，意思是args超参数字典等于右侧overrides, custom, kwargs三个超参数字典的并集
根据python语法，越靠右侧的字典优先级越高。
"""
args = {**overrides, **custom, **kwargs, "mode": "train", "session": self.session} 

- 根据这行代码可知：当三个超参数字典里有重复的超参数时，
  - `kwargs`（开发者自己定义的）覆盖
  - `custom`（ultralytics自己推测的应该有的超参数）覆盖
  - `overrides`(来自已训练好的权重文件`.pt`)

- `custom`和`overrides`都是在`train`内部被创建的，具体创建逻辑见下方代码：

In [ ]:
"""
这段代码来自train()方法内部
"""
overrides = YAML.load(checks.check_yaml(kwargs["cfg"])) if kwargs.get("cfg") else self.overrides
custom = {
    # NOTE: handle the case when 'cfg' includes 'data'.
    "data": (overrides.get("data") if kwargs.get("cfg") else None)
    or DEFAULT_CFG_DICT["data"]
    or TASK2DATA[self.task],
    "model": self.overrides["model"],
    "task": self.task,
}  # method defaults

- 根据上方代码可知，当传入`kwargs`里有`cfg`（cfg是一个可选参数，值是一份训练超参数YAML的路径）时会：
  - `check_yaml(kwargs["cfg"])`:
    - 1、值是普通本地路径或文件名时，检查后缀，不是`.yaml`或`.yml`或磁盘上对应路径没有文件跳到第三步，若都满足则返回路径。
    - 2、值是 `http://`、`https://`、`gs://`、`ul://` 这类地址时就下载到本地，返回下载后的路径。
    - 3、在ultralytics安装目录里按文件名搜索。找到唯一一个就返回它；一个都没有，或多个同名，抛出`FileNotFoundError`。
  - `overrides = YAML.load(xxx)`:
    - 当`check_yaml(kwargs["cfg"])`通过后会采用`YAML.load(xxx)`将`cfg`对应的`.yaml`超参数读取为字典并写入`overrides`。

<br>

- 当传入`kwargs`里没有`cfg`时会：
  - `overrides = self.overrides`:
    - `self.overrides`是在`model.train()`方法被调用前，模型初始化阶段就确定的超参数，数据来源是`.pt`或`.yaml`。

<br>

- 根据上方代码可知，`custom`只会构造`data`、`model`、`task`这3种超参数。
  - `data`:`(overrides.get("data") if kwargs.get("cfg") else None) or DEFAULT_CFG_DICT["data"] or TASK2DATA[self.task]`
    - 按 Python 短路 `or` 从前到后取第一个非空值：
    - 1、当`kwargs`里有`cfg`时，先尝试取 `overrides.get("data")`（即`cfg`对应`.yaml`文件里的`data`），防止后续`custom`覆盖`overrides`时把用户在`cfg`里配置好的`data`覆盖掉。
    - 2、当`kwargs`里没有`cfg`时，三元表达式故意返回`None`，不沿用`.pt`权重自带的`self.overrides["data"]`，避免直接调用`model.train()`时误用原预训练权重的大数据集（如`coco.yaml`）。
    - 3、若前一项为`None`（没传`cfg`，或`cfg`文件里未配置`data`），则依次回落到 `DEFAULT_CFG_DICT["data"]`（使用`default.yaml`中的`data`，默认为`None`），再回落到 `TASK2DATA[self.task]`（按当前任务类型自动匹配的内置轻量示例数据集，如目标检测任务对应`coco8.yaml`，轻量示例数据集由ultralytics提供，在需要被首次使用时联网下载）。
    <br>
  - `model`: `model = self.overrides["model"]`
    - 这里保证即使`kwargs`里有`cfg`导致`overrides`的`model`信息被改为对应的`.yaml`信息，`custom`构造的`model`仍然是模型初始化阶段的`model`信息。
    <br>
  - `task`: `task = self.task`
    - `self.task`在模型初始化阶段就被确定，放进`custom`同样可以避免被`cfg`文件里的`task`意外覆盖。
